In [0]:
dbutils.widgets.text("catalog","fk")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_date AS
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    d AS calendar_date,
    date_format(d, 'EEEE') AS day_name,
    weekofyear(d) AS iso_week,
    dayofweek(d) IN (1, 7) AS is_weekend
FROM (
    SELECT explode(
        sequence(DATE'2026-03-01', DATE'2026-03-31', INTERVAL 1 DAY)
    ) AS d
);          
          
""")

#store

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_store AS
SELECT * FROM {catalog}.silver.stores;          
          
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_product AS
SELECT
    product_id, product_name, brand, category_l1, category_l2,
    pack_size, pack_unit, mrp, selling_price, is_active
FROM {catalog}.silver.products;          
          
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_customer AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY customer_id, effective_from
    ) AS customer_sk,
    *
FROM {catalog}.silver.customers;          
          
""")

In [0]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gold.fact_orders AS
SELECT
    o.order_id,
    CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    c.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    o.channel,
    COUNT(i.order_id) AS line_count,
    COALESCE(SUM(i.quantity), 0) AS units,
    CAST(COALESCE(SUM(i.gross_amount), 0) AS DECIMAL(12,2)) AS gross_amount
FROM {catalog}.silver.orders o
LEFT JOIN {catalog}.silver.order_items i
    ON o.order_id = i.order_id
LEFT JOIN {catalog}.gold.dim_customer c
    ON o.customer_id = c.customer_id
   AND o.order_date >= c.effective_from
   AND (c.effective_to IS NULL OR o.order_date < c.effective_to)
GROUP BY
    o.order_id, o.order_date, c.customer_sk, o.customer_id,
    o.store_id, o.order_status, o.payment_method, o.channel;       
          
          
""")

In [0]:
spark.sql(f"""
   CREATE OR REPLACE TABLE {catalog}.gold.fact_order_line AS
SELECT
    i.order_id,
    i.line_number,
    CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    c.customer_sk,
    o.store_id,
    i.product_id,
    o.order_status,
    i.quantity,
    i.unit_price,
    i.gross_amount,
    i.discount_amount,
    i.net_amount
FROM {catalog}.silver.order_items i
JOIN {catalog}.silver.orders o
    ON i.order_id = o.order_id
LEFT JOIN {catalog}.gold.dim_customer c
    ON o.customer_id = c.customer_id
   AND o.order_date >= c.effective_from
   AND (c.effective_to IS NULL OR o.order_date < c.effective_to);       
          
          
          """)

kpi


In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE {catalog}.gold.agg_daily_store_sales AS
WITH daily AS (
    SELECT
        f.order_date_key,
        f.store_id,
        SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
        SUM(CASE WHEN f.order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders
    FROM {catalog}.gold.fact_orders f
    GROUP BY f.order_date_key, f.store_id
)
SELECT * FROM daily;
""")

In [0]:
spark.sql(f"""
          CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS
WITH k AS (
    SELECT
        SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
        SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders
    FROM {catalog}.silver.orders
)
SELECT
    delivered_orders,
    cancelled_orders
FROM k;
          
          """)

In [0]:
spark.sql(f"""
       CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
    run_ts TIMESTAMP,
    check_name STRING,
    source_value DECIMAL(18,2),
    target_value DECIMAL(18,2),
    status STRING
);   
          
""")